In [0]:
base="/Volumes/workspace/home_credit_landing/raw_files/"
tables=["application_train", "previous_application", "bureau", "credit_card_balance"]

for t in tables:
    df=spark.read.csv(f"{base}/{t}.csv",header=True,inferSchema=True)
    print(f"---{t}: {df.count():,} rows,{len(df.columns)} columns")
    df.printSchema()


In [0]:
base = "/Volumes/workspace/home_credit_landing/raw_files"
tables = ["application_train", "previous_application", "bureau", "credit_card_balance"]

for t in tables:
    dbutils.fs.mkdirs(f"{base}/{t}")
    dbutils.fs.mv(f"{base}/{t}.csv", f"{base}/{t}/{t}.csv")

display(dbutils.fs.ls(base))

In [0]:
df = spark.read.csv(f"{base}/application_train/application_train.csv", header=True)
print(df.count(), df.select("SK_ID_CURR").distinct().count())

df = spark.read.csv(f"{base}/previous_application/previous_application.csv", header=True)
print(df.count(), df.select("SK_ID_PREV").distinct().count())

df = spark.read.csv(f"{base}/bureau/bureau.csv", header=True)
print(df.count(), df.select("SK_ID_BUREAU").distinct().count())

cc = spark.read.csv(f"{base}/credit_card_balance/credit_card_balance.csv", header=True)
print(cc.count(), cc.select("SK_ID_PREV", "MONTHS_BALANCE").distinct().count())

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.home_credit_bronze;
CREATE SCHEMA IF NOT EXISTS workspace.home_credit_silver;
CREATE SCHEMA IF NOT EXISTS workspace.home_credit_gold;

In [0]:
%sql
SHOW SCHEMAS IN workspace LIKE 'home_credit*';

In [0]:
%sql
SELECT 'application_train' AS t, COUNT(*) AS n FROM workspace.home_credit_bronze.application_train
UNION ALL SELECT 'previous_application', COUNT(*) FROM workspace.home_credit_bronze.previous_application
UNION ALL SELECT 'bureau', COUNT(*) FROM workspace.home_credit_bronze.bureau
UNION ALL SELECT 'credit_card_balance', COUNT(*) FROM workspace.home_credit_bronze.credit_card_balance;

In [0]:
%sql
SELECT 'application_train' AS t, COUNT(*) AS rescued_rows
FROM workspace.home_credit_bronze.application_train WHERE _rescued_data IS NOT NULL
UNION ALL
SELECT 'previous_application', COUNT(*)
FROM workspace.home_credit_bronze.previous_application WHERE _rescued_data IS NOT NULL
UNION ALL
SELECT 'bureau', COUNT(*)
FROM workspace.home_credit_bronze.bureau WHERE _rescued_data IS NOT NULL
UNION ALL
SELECT 'credit_card_balance', COUNT(*)
FROM workspace.home_credit_bronze.credit_card_balance WHERE _rescued_data IS NOT NULL;

In [0]:
%sql
SELECT 'application_train' AS t,
       COUNT(*) FILTER (WHERE _source_file IS NULL OR _ingested_at IS NULL) AS missing_metadata
FROM workspace.home_credit_bronze.application_train
UNION ALL
SELECT 'previous_application',
       COUNT(*) FILTER (WHERE _source_file IS NULL OR _ingested_at IS NULL)
FROM workspace.home_credit_bronze.previous_application
UNION ALL
SELECT 'bureau',
       COUNT(*) FILTER (WHERE _source_file IS NULL OR _ingested_at IS NULL)
FROM workspace.home_credit_bronze.bureau
UNION ALL
SELECT 'credit_card_balance',
       COUNT(*) FILTER (WHERE _source_file IS NULL OR _ingested_at IS NULL)
FROM workspace.home_credit_bronze.credit_card_balance;

In [0]:
%sql
SELECT *
FROM workspace.home_credit_bronze.application_train LIMIT 5;